# ATV04 — Importância das Variáveis por Remoção (LOFO) · versão **KeyRecs**

**Disciplina:** Aprendizado de Máquina Explicável — Prof. Dr. Anderson Ara (DEST/LEG/UFPR)
**Grupo:** Débora Zanoello · Marcelo Carvalho Rezende Faria · Allysson Souza · Alex St Clair

Este notebook reproduz **todos os números e figuras** do deck *LOFO — Importância por Remoção (KeyRecs)*.

| # | Teste | Pergunta | Slide |
|---|---|---|---|
| 1 | LOFO na Random Forest (CV 5×10) | Que variáveis trazem informação **única**? | 8 |
| 2 | Teste t corrigido de Nadeau & Bengio | A importância difere de zero? | 8–9 |
| 3 | Estabilidade entre sementes | O ranking se mantém se mudarmos as partições? | 9 |
| 4 | LOFO na regressão linear | A importância depende do algoritmo? | 10 |
| 5 | Variável redundante: LOFO individual, LOFO por grupo e PFI | O que acontece com variáveis correlacionadas? | 11 |

**Definição.** Para uma perda $L$ avaliada fora do treino,

$$\mathrm{LOFO}_j = \mathbb{E}\big[L(Y, \hat f_{-j}(X_{-j}))\big] - \mathbb{E}\big[L(Y, \hat f(X))\big],$$

em que $\hat f_{-j}$ é o **mesmo algoritmo re-treinado sem** $X_j$ (Lei et al., 2018; Williamson et al., 2023; Molnar, 2025).

> ⚠️ O KeyRecs **não tem rótulo clínico**. As explicações descrevem o comportamento do modelo, não causas, e nada aqui diagnostica.

## 0 · Ambiente

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, RepeatedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import sklearn

warnings.filterwarnings("ignore")
SEMENTE = 42
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__)

COR = {"sup": "#F7F6F1", "tinta": "#1B2233", "tinta2": "#4A5263", "azul": "#2A68C2",
       "lar": "#D9572B", "amar": "#E0A400", "cinza": "#9AA0AC", "grade": "#E2E0D8"}
mpl.rcParams.update({"figure.facecolor": COR["sup"], "axes.facecolor": COR["sup"],
    "axes.grid": True, "axes.axisbelow": True, "grid.color": COR["grade"],
    "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "font.size": 11, "legend.frameon": False})

## 1 · Dados

Tabela analítica da ATV02/ATV03 (uma linha por participante), salva em `Atividade 3/saidas/tabela_analitica.csv`.
Coloque o arquivo na mesma pasta deste notebook.

- **Resposta** `hesitacao_livre`: mediana do *flight time* (DD − dwell da tecla anterior) no **texto livre**, em segundos.
- **7 explicativas:** 4 da tarefa **motora** de senha fixa (`fix_dwell_med`, `fix_flight_med`, `fix_total_cv`, `fix_ganho_pratica`) + `dem_idade`, `dem_genero`, `dem_lateralidade`.
- `fix_total_med` fica **fora** do modelo principal (colinear com dwell + flight, decisão da ATV02); ele volta só no Teste 5.

In [ ]:
dados = pd.read_csv("tabela_analitica.csv", index_col="participant")
ALVO = "hesitacao_livre"
FEATS = ["fix_dwell_med", "fix_flight_med", "fix_total_cv", "fix_ganho_pratica",
         "dem_idade", "dem_genero", "dem_lateralidade"]
CATS = {"dem_genero": "Male", "dem_lateralidade": "Right-Handed"}   # referência = categoria mais frequente

print("n =", len(dados))
display(dados[FEATS + ["fix_total_med", ALVO]].describe(include="all").T.iloc[:, :6])

In [ ]:
def desenho(X):
    # Dummy 0/1 com referência fixa. Cada categórica tem 2 níveis, então 1 coluna = 1 variável.
    X = X.copy()
    for c, ref in CATS.items():
        if c in X:
            X[c] = (X[c] != ref).astype(float)
    return X.astype(float)

X_all = desenho(dados[FEATS + ["fix_total_med"]])
y_all = dados[ALVO]
print(X_all[FEATS].corr().round(2))

## 2 · Modelo e protocolo de validação

- **Random Forest**, 300 árvores. `min_samples_leaf` e `max_features` escolhidos por CV interna **no treino** (mesma grade da ATV03).
- **LOFO** por **validação cruzada 5 dobras × 10 repetições**: as **mesmas 50 dobras** para o modelo completo e para cada variável retirada. Assim, as diferenças são **pareadas** dobra a dobra.
- Métrica: **RMSE** (relatado em ms).

In [ ]:
def novo_rf(**hp):
    return RandomForestRegressor(n_estimators=300, random_state=SEMENTE, n_jobs=-1, **hp)

def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))

X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.30, random_state=SEMENTE)
t0 = time.time()
busca = GridSearchCV(novo_rf(), {"min_samples_leaf": [3, 5, 10, 15, 20], "max_features": [0.4, 0.7, 1.0]},
                     cv=RepeatedKFold(n_splits=5, n_repeats=2, random_state=SEMENTE),
                     scoring="neg_root_mean_squared_error", n_jobs=1).fit(X_tr[FEATS], y_tr)
HP = busca.best_params_
print("Hiperparâmetros da floresta:", HP, f"({time.time()-t0:.0f} s)")

DOBRAS = list(RepeatedKFold(n_splits=5, n_repeats=10, random_state=SEMENTE).split(X_all))
print("Dobras de CV:", len(DOBRAS))

## 3 · Implementação do LOFO e do teste de significância

**Algoritmo** (para cada dobra $k$):
1. treinar $\hat f$ com todas as variáveis e medir $L_k$ na validação;
2. para cada variável (ou **grupo**) $j$: re-treinar o **mesmo algoritmo, com os mesmos hiperparâmetros**, sem $X_j$ e medir $L_k^{(-j)}$;
3. $\Delta_{jk} = L_k^{(-j)} - L_k$; resumir com média e dispersão entre dobras.

**Teste t corrigido (Nadeau & Bengio, 2003).** As dobras de CV repetida compartilham observações, então o teste t comum subestima a variância. A correção usa

$$t = \frac{\bar\Delta_j}{\sqrt{\left(\tfrac1K + \tfrac{n_\text{teste}}{n_\text{treino}}\right) s^2_{\Delta_j}}}, \qquad \tfrac{n_\text{teste}}{n_\text{treino}} = \tfrac{1}{4} \text{ em 5 dobras},$$

com $K-1$ graus de liberdade.

In [ ]:
def teste_nb(d, razao=0.25):
    d = np.asarray(d); k = len(d)
    t = d.mean() / np.sqrt((1 / k + razao) * d.var(ddof=1))
    return t, 2 * stats.t.sf(abs(t), k - 1)


def lofo(fabrica, feats, grupos=None, dobras=None):
    # LOFO por CV. Retorna (tabela resumo, RMSE base por dobra, dict de Δ por dobra).
    dobras = dobras or DOBRAS
    grupos = grupos or {f: [f] for f in feats}
    base, sem = [], {g: [] for g in grupos}
    for tr, te in dobras:
        Xa, Xb, ya, yb = X_all.iloc[tr], X_all.iloc[te], y_all.iloc[tr], y_all.iloc[te]
        base.append(rmse(yb, fabrica().fit(Xa[feats], ya).predict(Xb[feats])))
        for g, cols in grupos.items():
            fs = [f for f in feats if f not in cols]
            sem[g].append(rmse(yb, fabrica().fit(Xa[fs], ya).predict(Xb[fs])))
    base = np.array(base)
    deltas = {g: np.array(sem[g]) - base for g in grupos}
    linhas = []
    for g, d in deltas.items():
        t, p = teste_nb(d)
        linhas.append({"variavel": g, "lofo_ms": 1000 * d.mean(), "dp_ms": 1000 * d.std(ddof=1),
                       "pct_rmse": 100 * d.mean() / base.mean(), "dobras_pos": (d > 0).mean(), "t": t, "p_NB": p})
    tab = pd.DataFrame(linhas).sort_values("lofo_ms", ascending=False).reset_index(drop=True)
    return tab, base, deltas

## Teste 1 · LOFO na Random Forest

In [ ]:
t0 = time.time()
tab_rf, base_rf, d_rf = lofo(lambda: novo_rf(**HP), FEATS)
base_med = np.array([rmse(y_all.iloc[te], np.full(len(te), y_all.iloc[tr].median())) for tr, te in DOBRAS])
print(f"RMSE da floresta: {1000*base_rf.mean():.1f} ms | prever a mediana: {1000*base_med.mean():.1f} ms "
      f"({time.time()-t0:.0f} s)")
display(tab_rf.round({"lofo_ms": 2, "dp_ms": 2, "pct_rmse": 1, "dobras_pos": 2, "t": 2, "p_NB": 4}))

In [ ]:
ROT = {"fix_flight_med": "flight (pausa entre teclas)", "fix_total_cv": "CV do tempo total",
       "fix_dwell_med": "dwell (tecla pressionada)", "fix_ganho_pratica": "ganho com prática",
       "dem_idade": "idade", "dem_genero": "gênero", "dem_lateralidade": "lateralidade",
       "fix_total_med": "tempo total"}

t = tab_rf.sort_values("lofo_ms")
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh([ROT[v] for v in t["variavel"]], t["lofo_ms"], xerr=1.96 * t["dp_ms"] / np.sqrt(len(DOBRAS)),
        color=[COR["lar"] if v > 0 else COR["azul"] for v in t["lofo_ms"]], height=0.6,
        error_kw={"capsize": 3})
ax.axvline(0, color=COR["tinta"], lw=1); ax.grid(axis="y", visible=False)
ax.set_xlabel("aumento do RMSE ao remover a variável (ms) — IC 95% da média entre dobras")
ax.set_title("Teste 1 · LOFO — Random Forest (CV 5×10)")
plt.show()

**Leitura.** Só o `fix_flight_med` carrega informação **única**: sem ele, o RMSE sobe ≈ 10 ms (≈ 30%). As demais ficam em torno de zero. `fix_dwell_med` e `dem_idade` são levemente negativas: a floresta generaliza um pouco melhor **sem** elas.

## Teste 2 · Incerteza dobra a dobra

In [ ]:
ordem = tab_rf["variavel"].tolist()[::-1]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.boxplot([1000 * d_rf[v] for v in ordem], vert=False, widths=0.55)
ax.set_yticks(range(1, len(ordem) + 1), [ROT[v] for v in ordem])
ax.axvline(0, color=COR["tinta"], lw=1); ax.grid(axis="y", visible=False)
ax.set_xlabel("Δ RMSE por dobra (ms) — 50 dobras")
ax.set_title("Teste 2 · LOFO dobra a dobra: sinal × ruído")
plt.show()

print("Variáveis com p < 0,05 (Nadeau & Bengio):", tab_rf.loc[tab_rf["p_NB"] < 0.05, "variavel"].tolist())

**Leitura.** Re-treinar introduz ruído (o dp do flight é metade da média), mas o flight é positivo em ≈ 96% das dobras e é a **única** variável com p < 0,05 no teste corrigido.

## Teste 3 · Estabilidade do ranking entre partições

Repetimos o LOFO com **5 sementes** de partição diferentes (5 dobras × 2 repetições cada) e comparamos a posição de cada variável.

In [ ]:
t0 = time.time()
ranks = []
for s in [1, 7, 21, 99, 2026]:
    dob = list(RepeatedKFold(n_splits=5, n_repeats=2, random_state=s).split(X_all))
    tb, _, _ = lofo(lambda: novo_rf(**HP), FEATS, dobras=dob)
    ranks.append(tb.set_index("variavel")["lofo_ms"].rank(ascending=False).rename(f"semente {s}"))
rank_tab = pd.concat(ranks, axis=1).loc[tab_rf["variavel"]].astype(int)
rank_tab["faixa"] = rank_tab.min(axis=1).astype(str) + "º–" + rank_tab.max(axis=1).astype(str) + "º"
print(f"({time.time()-t0:.0f} s)")
display(rank_tab)

**Leitura.** `fix_flight_med` fica em 1º e `fix_total_cv` em 2º nas 5 partições. Da 3ª posição para baixo a ordem muda a cada semente: a cauda do ranking é ruído e não deve ser interpretada.

## Teste 4 · Dependência do algoritmo: regressão linear

Mesmas 50 dobras, outro algoritmo (OLS; contínuas padronizadas no treino de cada dobra).

In [ ]:
CONT = ["fix_dwell_med", "fix_flight_med", "fix_total_cv", "fix_ganho_pratica", "dem_idade", "fix_total_med"]

def novo_lin():
    ct = ColumnTransformer([("z", StandardScaler(), lambda X: [c for c in X.columns if c in CONT])],
                           remainder="passthrough")
    return make_pipeline(ct, LinearRegression())

tab_lin, base_lin, _ = lofo(novo_lin, FEATS)
print(f"RMSE linear: {1000*base_lin.mean():.1f} ms")
comp = (tab_rf.set_index("variavel")[["lofo_ms", "p_NB"]]
        .join(tab_lin.set_index("variavel")[["lofo_ms", "p_NB"]], lsuffix="_RF", rsuffix="_linear"))
comp["rank_RF"] = comp["lofo_ms_RF"].rank(ascending=False).astype(int)
comp["rank_linear"] = comp["lofo_ms_linear"].rank(ascending=False).astype(int)
display(comp.round(3))

In [ ]:
ordem = tab_rf.sort_values("lofo_ms")["variavel"].tolist()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, tb, tt in [(axes[0], tab_rf, "Random Forest"), (axes[1], tab_lin, "Regressão linear")]:
    v = tb.set_index("variavel").loc[ordem, "lofo_ms"]
    ax.barh([ROT[x] for x in ordem], v, color=[COR["lar"] if x > 0 else COR["azul"] for x in v], height=0.6)
    ax.axvline(0, color=COR["tinta"], lw=1); ax.grid(axis="y", visible=False)
    ax.set_title(f"Teste 4 · LOFO — {tt}"); ax.set_xlabel("Δ RMSE (ms)")
plt.tight_layout(); plt.show()

**Leitura.** O flight domina nos dois, mas pesa **mais** no linear. Na cauda a ordem muda (gênero sobe para 2º). O LOFO explica o **par algoritmo + dados**, não um valor intrínseco da variável — daí a versão populacional de Williamson et al. (2023).

## Teste 5 · Variável redundante: LOFO individual, LOFO por grupo e PFI

Experimento proposital: devolvemos ao modelo `fix_total_med` (tempo total da senha), fortemente correlacionado com o flight.
Comparamos:
- **LOFO individual** de flight e de tempo total;
- **LOFO por grupo** {flight, tempo total} — as duas removidas juntas;
- **PFI** (importância por permutação) no hold-out de 30%, com e sem o tempo total.

In [ ]:
rho = dados[["fix_flight_med", "fix_total_med"]].corr().iloc[0, 1]
print(f"Correlação flight × tempo total: ρ = {rho:.2f}")

FEATS_R = FEATS + ["fix_total_med"]
grupos_r = {f: [f] for f in FEATS_R}
grupos_r["{flight + total_med}"] = ["fix_flight_med", "fix_total_med"]
t0 = time.time()
tab_r, _, _ = lofo(lambda: novo_rf(**HP), FEATS_R, grupos_r)
print(f"({time.time()-t0:.0f} s)")
display(tab_r.round({"lofo_ms": 2, "dp_ms": 2, "pct_rmse": 1, "dobras_pos": 2, "t": 2, "p_NB": 4}))

In [ ]:
def pfi(feats):
    m = novo_rf(**HP).fit(X_tr[feats], y_tr)
    r = permutation_importance(m, X_te[feats], y_te, scoring="neg_root_mean_squared_error",
                               n_repeats=50, random_state=SEMENTE)
    return pd.Series(1000 * r.importances_mean, index=feats)

pfi_7, pfi_8 = pfi(FEATS), pfi(FEATS_R)
resumo5 = pd.DataFrame({
    "LOFO 7 var. (ms)": tab_rf.set_index("variavel")["lofo_ms"],
    "LOFO 8 var. (ms)": tab_r.set_index("variavel")["lofo_ms"],
    "PFI 7 var. (ms)": pfi_7, "PFI 8 var. (ms)": pfi_8,
}).loc[["fix_flight_med", "fix_total_med", "{flight + total_med}"]]
display(resumo5.round(2))
queda = 1 - tab_r.set_index("variavel").loc["fix_flight_med", "lofo_ms"] / tab_rf.set_index("variavel").loc["fix_flight_med", "lofo_ms"]
print(f"Queda do LOFO do flight ao incluir a substituta: {100*queda:.0f}%")

In [ ]:
l7 = tab_rf.set_index("variavel")["lofo_ms"]; l8 = tab_r.set_index("variavel")["lofo_ms"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5)); w = 0.38
ax = axes[0]
ax.bar([-w/2], [l7["fix_flight_med"]], w, color=COR["cinza"], label="7 variáveis")
ax.bar(np.arange(3) + w/2, [l8["fix_flight_med"], l8["fix_total_med"], l8["{flight + total_med}"]], w,
       color=[COR["lar"], COR["lar"], COR["amar"]], label="+ tempo total")
ax.set_xticks(range(3), ["flight", "tempo total", "grupo"]); ax.legend(); ax.grid(axis="x", visible=False)
ax.set_title("LOFO: a importância se divide"); ax.set_ylabel("Δ RMSE (ms)")
ax = axes[1]
ax.bar([-w/2], [pfi_7["fix_flight_med"]], w, color=COR["cinza"], label="7 variáveis")
ax.bar(np.arange(2) + w/2, [pfi_8["fix_flight_med"], pfi_8["fix_total_med"]], w, color=COR["azul"], label="+ tempo total")
ax.set_xticks(range(2), ["flight", "tempo total"]); ax.legend(); ax.grid(axis="x", visible=False)
ax.set_title("PFI: o modelo treinado nem percebe"); ax.set_ylabel("Δ RMSE ao permutar (ms)")
plt.tight_layout(); plt.show()

**Leitura.** Com a substituta no modelo, o LOFO do flight cai ≈ 74% e o do tempo total fica perto de zero: **cada uma cobre a outra**. Removidas juntas, a perda volta ao nível original — a informação está no **grupo**. A PFI quase não muda, porque o modelo treinado continua usando o flight. As duas medidas estão certas, mas respondem a perguntas diferentes (Hooker, Mentch & Zhou, 2021).

## Resumo

| Teste | Resultado |
|---|---|
| 1 · LOFO RF | flight é a única variável com informação única (≈ +10 ms, ≈ 30% do RMSE) |
| 2 · Nadeau & Bengio | só o flight tem p < 0,05; positivo em ≈ 96% das dobras |
| 3 · Sementes | flight 1º e CV do tempo total 2º em 5/5; cauda instável |
| 4 · Linear | mesma variável principal, peso maior; cauda reordenada |
| 5 · Redundância | LOFO divide a importância entre substitutas; o grupo recupera; PFI não muda |

### Referências
Lei et al. (2018), *JASA* 113(523) · Williamson et al. (2023), *JASA* 118(543) · Hooker, Mentch & Zhou (2021), *Statistics and Computing* 31:82 · Covert, Lundberg & Lee (2021), *JMLR* 22(209) · Fisher, Rudin & Dominici (2019), *JMLR* 20(177) · Molnar (2025), *Interpretable Machine Learning*, 3ª ed. · Nadeau & Bengio (2003), *Machine Learning* 52 · Dias et al. (2023), KeyRecs, Zenodo 10.5281/zenodo.7886743.